<div dir="rtl">
<h1>نشانی هر خانهٔ جدول مقایسه</h1>
<p>درس 35 از 76 · امتیاز همهٔ جفت‌ها را دستی بسازیم · <code dir="ltr">29-scores</code></p>
<p><a target="_self" href="http://127.0.0.1:8000/part-05/chapter-02/29-scores.html">📖 بازگشت به همین درس</a></p>
<p>امتیاز همهٔ جفت‌های Query و Key را با حلقه بسازید و با matmul بسنجید.</p><p>پیش‌نیاز: ضرب داخلی و تفاوت محور موقعیت با محور ویژگی را بشناسید.</p>
<p>این دفتر نیمهٔ عملی درس است. مثال‌ها آمادهٔ اجرا هستند؛ دو Cell با برچسب TODO را خودتان کامل کنید. پیام INCOMPLETE یعنی هنوز چیزی ننوشته‌اید، نه اینکه پاسخ درست است. جواب مرجع در این دفتر پنهان نشده است.</p>
<p>از بالا به پایین اجرا کنید. پس از تغییر هر تابع، Cell آن و سپس Cell آزمون را دوباره اجرا کنید. برای بررسی نهایی، از منوی <code>Kernel → Restart Kernel and Run All Cells</code> استفاده کنید.</p>
</div>

In [ ]:
from pathlib import Path
import os
import sys

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "mini_gpt").is_dir() and (p / "book_src").is_dir()), None)
if project_root is None:
    raise RuntimeError("Extract the complete learning project; open this notebook inside it.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("Python:", sys.executable)
print("Project:", project_root)

<div dir="rtl">
<h2>قبل از اجرا، پیش‌بینی کنید</h2>
<p>سه Query دوویژگی و چهار Key دوویژگی داریم. جدول چند سطر و چند ستون دارد؟ خانهٔ [1,2] دقیقاً کدام دو بردار را مقایسه می‌کند؟</p>
</div>

<div dir="rtl"><p>پیش‌بینی من: …</p></div>

In [ ]:
import math
import torch
torch.set_num_threads(1)
torch.manual_seed(17)
q = torch.tensor([[1.,0.],[0.,1.],[1.,2.]])
k = torch.tensor([[1.,1.],[0.,1.],[2.,-1.],[3.,0.]])
print('Q:',q,'K:',k)

<div dir="rtl">
<h2>این بار شما کد بنویسید</h2>
<p>تابع pair_scores(q, k) را با حلقه روی Query و Key بنویسید؛ هر خانه مجموع حاصل‌ضرب ویژگی‌هاست. از q@k.T در راه‌حل این بخش استفاده نکنید. خروجی Tensor شکل (Tq,Tk) باشد.</p>
</div>

In [ ]:
def pair_scores(q, k):
    # TODO
    return None

In [ ]:
def test_exercise():
    result = pair_scores(q,k)
    if result is None: return False
    assert result.shape == (3,4)
    torch.testing.assert_close(result,q@k.T)
    assert result[1,2].item() == -1.
    for a,b in ((q[:1],k[:2]),(torch.ones(2,3),torch.arange(12.).reshape(4,3))):
        torch.testing.assert_close(pair_scores(a,b),a@b.T)
    return True

exercise_complete = test_exercise()
print("PASS" if exercise_complete else "INCOMPLETE: complete the TODO first")

<div dir="rtl">
<h2>فقط یک عامل را تغییر دهید</h2>
<p>فقط Key سوم را در منفی یک ضرب کنید. پیش‌بینی کنید کدام ستون جدول علامت عوض می‌کند؛ Queryها و بقیهٔ Keyها ثابت‌اند.</p>
</div>

In [ ]:
changed_k = k.clone()
changed_k[2] *= -1
print('score difference:',q@changed_k.T-q@k.T)

<div dir="rtl">
<h2>خرابی را پیدا کنید</h2>
<p>حذف Transpose در مثال مربعی گاهی خطا نمی‌دهد. تابع score_matrix(q, k) را اصلاح کنید؛ مثال نامربعی نیز باید کار کند.</p>
</div>

In [ ]:
square_q = torch.eye(2)
square_k = torch.tensor([[1.,2.],[3.,4.]])
print('same shape, wrong meaning:',square_q@square_k)
try:
    print(q@k)
except RuntimeError as error:
    print('expected incompatible axes:',error)

<div dir="rtl">
<h2>اصلاح را خودتان بنویسید</h2>
<p>علت را توضیح دهید، سپس تابع زیر را کامل کنید. خطای عمدی بالا یک نمونهٔ آموزشی است؛ آزمون پایین باید اصلاح شما را بسنجد.</p>
</div>

In [ ]:
def score_matrix(q, k):
    # TODO
    return None

In [ ]:
def test_repair():
    result = score_matrix(q,k)
    if result is None: return False
    torch.testing.assert_close(result,q@k.T)
    torch.testing.assert_close(score_matrix(square_q,square_k),square_q@square_k.T)
    return True

repair_complete = test_repair()
print("PASS" if repair_complete else "INCOMPLETE: complete the TODO first")

<div dir="rtl">
<h2>در Mini-GPT کجا به کار می‌آید؟</h2>
<p>raw_scores در mini_gpt/attention.py همین مقایسه‌هاست؛ در مدل، محورهای Batch و Head جلو این دو محور قرار می‌گیرند. هنوز Softmax و Mask اعمال نشده‌اند.</p>
</div>

<div dir="rtl">
<h2>با زبان خودتان توضیح دهید</h2>
<p>چرا موفق‌بودن ضرب دو ماتریس مربعی برای تأیید معنای محورهای Attention کافی نیست؟</p>
</div>
<div dir="rtl"><p>پیش‌بینی و مشاهدهٔ من: …</p><p>علت خرابی و اصلاح من: …</p></div>

<div dir="rtl"><p><a target="_self" href="http://127.0.0.1:8000/part-05/chapter-02/29-scores.html">بازگشت به درس و ادامهٔ مسیر</a> · <a target="_self" href="http://127.0.0.1:8000/answers/29-scores.html#lab-solution">فقط پس از تلاش: راه‌حل مرجع آزمایشگاه</a></p></div>